In [1]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM

/home/zaccosenza/code/project-enzyme/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv('../data/01_raw/train_raw.csv')
print(df.shape)
df.head(2)

(31390, 5)


,seq_id,protein_sequence,pH,data_source,tm
0,0,AAAAKAAALALLGEAPEVVDIWLPAGWRQPFRVFRLERKGDGVLVG...,7.0,doi.org/10.1038/s41592-020-0801-4,75.7
1,1,AAADGEPLHNEEERAGAGQVGRSLPQESEEQRTGSRPRRRRDLGSR...,7.0,doi.org/10.1038/s41592-020-0801-4,50.5


In [3]:
MODEL_NAME = 'facebook/esm2_t33_650M_UR50D'

if torch.xpu.is_available():
    device = torch.device('xpu')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print('device:', device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME).to(device)
model.eval()

device: xpu


Loading weights: 100%|██████████| 539/539 [00:00<00:00, 1414.63it/s]


EsmForMaskedLM(
  (esm): EsmModel(
    (embeddings): EsmEmbeddings(
      (word_embeddings): Embedding(33, 1280, padding_idx=1)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (rotary_embeddings): EsmRotaryEmbedding()
    (encoder): EsmEncoder(
      (layer): ModuleList(
        (0-32): 33 x EsmLayer(
          (attention): EsmAttention(
            (self): EsmSelfAttention(
              (query): Linear(in_features=1280, out_features=1280, bias=True)
              (key): Linear(in_features=1280, out_features=1280, bias=True)
              (value): Linear(in_features=1280, out_features=1280, bias=True)
            )
            (output): EsmSelfOutput(
              (dense): Linear(in_features=1280, out_features=1280, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (LayerNorm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          )
          (intermediate): EsmIntermediate(
            (dense): Linear(in_features=1280, ou

In [4]:
def embed_sequence(seq: str) -> torch.Tensor:
    """Returns mean-pooled embedding vector of shape (1280,)."""
    inputs = tokenizer(seq, return_tensors='pt', truncation=True, max_length=1024).to(device)
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
    # last hidden state is the final encoder layer before the LM head
    return outputs.hidden_states[-1][0, 1:-1].mean(dim=0).cpu()

In [5]:
# smoke test on a single sequence
sample_seq = df['protein_sequence'].iloc[0]
vec = embed_sequence(sample_seq)
print('sequence length:', len(sample_seq))
print('embedding shape:', vec.shape)
print('embedding sample:', vec[:5])

sequence length: 341
embedding shape: torch.Size([1280])
embedding sample: tensor([ 0.0705, -0.1038, -0.0003, -0.0492, -0.0456])


## Zero-shot mutation effect scoring (log-likelihood ratio)

In [6]:
import torch.nn.functional as F

WILDTYPE = 'VPVNPEPDATSVENVALKTGSGDSQSDPIKADLEVKGQSALPFDVDCWAILCKGAPNVLQRVNEKTKNSNRDRSGANKGPFKDPQKWGIKALPPKNPSWSAQDFKSPEEYAFASSLQGGTNAILAPVNLASQNSQGGVLNGFYSANKVAQFDPSKPQQTKGTWFQITKFTGAAGPYCKALGSNDKSVCDKNKNIAGDWGFDPAKWAYQYDEKNNKFNYVGK'

def score_mutation(mutant_seq: str, wildtype: str = WILDTYPE) -> float:
    """
    Log-likelihood ratio score for a mutant sequence vs wildtype.
    For each mutated position, masks that position and computes:
        log p(mutant_aa | context) - log p(wildtype_aa | context)
    Returns the sum over all mutated positions.
    A negative score = evolutionarily unusual = likely destabilizing.
    """
    assert len(mutant_seq) == len(wildtype), "mutant and wildtype must be the same length"

    mutated_positions = [i for i, (w, m) in enumerate(zip(wildtype, mutant_seq)) if w != m]
    if not mutated_positions:
        return 0.0

    total_score = 0.0

    for pos in mutated_positions:
        # mask the mutated position (ESM tokenizer offset: +1 for BOS token)
        inputs = tokenizer(wildtype, return_tensors='pt', truncation=True, max_length=1024).to(device)
        input_ids = inputs['input_ids'].clone()
        token_pos = pos + 1  # +1 for BOS
        input_ids[0, token_pos] = tokenizer.mask_token_id

        with torch.no_grad():
            logits = model(**{**inputs, 'input_ids': input_ids}).logits  # (1, seq_len, vocab)

        log_probs = F.log_softmax(logits[0, token_pos], dim=-1)

        wt_token  = tokenizer.convert_tokens_to_ids(wildtype[pos])
        mut_token = tokenizer.convert_tokens_to_ids(mutant_seq[pos])

        total_score += (log_probs[mut_token] - log_probs[wt_token]).item()

    return total_score

In [7]:
# smoke test: score a single mutant vs the wildtype itself (should be 0.0)
wt_score = score_mutation(WILDTYPE)
print(f'wildtype score (expect 0.0): {wt_score}')

# score a single test sequence
import pandas as pd
test = pd.read_csv('../data/01_raw/test.csv').merge(
    pd.read_csv('../data/01_raw/test_labels.csv'), on='seq_id'
)
sample = test[test['protein_sequence'].str.len() == len(WILDTYPE)].iloc[0]
score = score_mutation(sample['protein_sequence'])
n_mut = sum(a != b for a, b in zip(sample['protein_sequence'], WILDTYPE))
print(f'sample sequence: {n_mut} mutation(s), LLR score={score:.4f}, actual Tm={sample["tm"]:.1f}°C')

wildtype score (expect 0.0): 0.0
sample sequence: 1 mutation(s), LLR score=0.1496, actual Tm=77.3°C


In [11]:
from scipy.stats import spearmanr
from tqdm import tqdm

# score all same-length test sequences and evaluate Spearman correlation
test_mut = test[test['protein_sequence'].str.len() == len(WILDTYPE)].copy()

scores = []
for seq in tqdm(test_mut['protein_sequence'], desc='scoring'):
    scores.append(score_mutation(seq))

test_mut['llr_score'] = scores

rho, pval = spearmanr(test_mut['llr_score'], test_mut['tm'])
print(f'Zero-shot LLR vs Tm — Spearman ρ={rho:.4f}, p={pval:.2e}, n={len(test_mut)}')

scoring: 100%|██████████| 2336/2336 [11:34<00:00,  3.36it/s]

Zero-shot LLR vs Tm — Spearman ρ=0.1105, p=8.50e-08, n=2336
